In [1]:
%pip install requests tqdm pillow ddgs 

  Using cached lxml-6.0.2-cp311-cp311-manylinux_2_26_aarch64.manylinux_2_28_aarch64.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 72.1 MB/s  0:00:00
Using cached lxml-6.0.2-cp311-cp311-manylinux_2_26_aarch64.manylinux_2_28_aarch64.whl (5.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ddgs]1/3 [lxml]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import csv
import requests
from tqdm import tqdm
from PIL import Image
from io import BytesIO

from ddgs import DDGS
import time

# ---------------------------
# PARAMETRES
# ---------------------------

DATASET_DIR = "../data/dataset_aromatic_plants"
CSV_FILE = "../data/dataset_aromatic_plants/dataset.csv"

IMAGES_PER_PLANT = 30
IMAGE_SIZE = (400, 400)

plants = {
    "basil": "Ocimum basilicum",
    "mint": "Mentha",
    "rosemary": "Salvia rosmarinus",
    "thyme": "Thymus vulgaris",
    "oregano": "Origanum vulgare",
    "parsley": "Petroselinum crispum",
    "chives": "Allium schoenoprasum",
    "sage": "Salvia officinalis",
    "lavender": "Lavandula",
    "dill": "Anethum graveolens",
    "tarragon": "Artemisia dracunculus",
    "coriander": "Coriandrum sativum",
    "lemongrass": "Cymbopogon citratus",
    "bay_leaf": "Laurus nobilis",
    "marjoram": "Origanum majorana",
    "savory": "Satureja hortensis",
    "fennel": "Foeniculum vulgare",
    "anise": "Pimpinella anisum",
    "stevia": "Stevia rebaudiana",
    "lemon_balm": "Melissa officinalis",
    "chamomile": "Matricaria chamomilla",
    "pandan": "Pandanus amaryllifolius",
    "holy_basil": "Ocimum tenuiflorum",
    "lemon_verbena": "Aloysia citrodora",
    "wintergreen": "Gaultheria procumbens",
    "mugwort": "Artemisia vulgaris",
    "hyssop": "Hyssopus officinalis",
    "angelica": "Angelica archangelica",
    "lovage": "Levisticum officinale",
    "borage": "Borago officinalis"
}

# ---------------------------
# CREATION DOSSIERS
# ---------------------------

os.makedirs(DATASET_DIR, exist_ok=True)

csv_rows = []

# ---------------------------
# TELECHARGEMENT + RESIZE
# ---------------------------

def download_and_resize(url, filepath):

    try:
        r = requests.get(url, timeout=15)

        img = Image.open(BytesIO(r.content)).convert("RGB")

        img = img.resize(IMAGE_SIZE)

        img.save(filepath, "JPEG", quality=90)

        return True

    except:
        return False

In [ ]:
# ---------------------------
# TELECHARGEMENT DATASET
# ---------------------------

for plant_name, taxon in plants.items():

    print(f"\nTéléchargement : {plant_name}")

    plant_dir = os.path.join(DATASET_DIR, plant_name)

    os.makedirs(plant_dir, exist_ok=True)

    count = 0
    page = 1

    pbar = tqdm(total=IMAGES_PER_PLANT)

    while count < IMAGES_PER_PLANT:

        url = "https://api.inaturalist.org/v1/observations"

        params = {
            "taxon_name": taxon,
            "quality_grade": "casual",
            "identified": "true",
            "photos": "true",
            "per_page": 200,
            "page": page
        }

        response = requests.get(url, params=params).json()

        results = response["results"]

        if not results:
            break

        for obs in results:

            for photo in obs["photos"]:

                img_url = photo["url"].replace("square", "large")

                filename = f"{count}.jpg"
                filepath = os.path.join(plant_dir, filename)

                success = download_and_resize(img_url, filepath)

                if success:

                    relative_path = os.path.join(plant_name, filename)

                    csv_rows.append([plant_name, relative_path])

                    count += 1
                    pbar.update(1)

                if count >= IMAGES_PER_PLANT:
                    break

            if count >= IMAGES_PER_PLANT:
                break

        page += 1

    pbar.close()


# ---------------------------
# ECRITURE CSV
# ---------------------------

with open(CSV_FILE, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow(["plant_type", "image_path"])

    writer.writerows(csv_rows)

print("\nDataset et CSV créés avec succès.")

In [3]:
"""
Scraping d'images d'aromates via DuckDuckGo Images
====================================================
Utilise le package `ddgs` (successeur de `duckduckgo_search`)

Installation :
    pip install ddgs Pillow requests

Arborescence produite :
    images/
      basilic_commun/
        basilic_commun_0001.jpg
        …
      dataset.csv
"""

# ─────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────



TARGET_COUNT = 150
IMAGE_SIZE   = (400, 400)
OUTPUT_DIR   = "dataset_aromatic_duckduckgo"
CSV_FILENAME = "dataset.csv"
DELAY_S      = 0.1        # pause entre téléchargements
TIMEOUT_S    = 10

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

# ─────────────────────────────────────────────
#  30 AROMATES
#  Format : (nom_commun_fr, [requêtes_ddg])
# ─────────────────────────────────────────────
AROMATES: list[tuple[str, list[str]]] = [
    ("thym commun",             ["thyme"]),
    ("romarin",                 ["rosemary"]),
    
    ("menthe poivree",          ["peppermint"]),
    ("menthe verte",            ["spearmint"]),
    ("origan",                  ["oregano"]),
    ("coriandre",               ["coriander"]),
    ("aneth",                   ["dill"]),
    ("persil",                  ["parsley"]),
    ("ciboulette",              ["chives"]),
    ("estragon",                ["tarragon"]),
]

#[
#    #("basilic commun",          ["basilic", "ocimum basilicum", "basil"]),
#    ("thym commun",             ["thym", "thymus vulgaris", "thyme"]),
#    ("romarin",                 ["romarin", "rosmarinus officinalis", "rosemary"]),
#    #("lavande vraie",           ["lavandula angustifolia", "lavender"]),
#    ("menthe poivree",          ["menthe", "mentha piperita", "peppermint"]),
#    ("menthe verte",            ["menthe", "mentha spicata", "spearmint"]),
#    ("origan",                  ["origan", "origanum vulgare", "oregano"]),
#    #("sauge officinale",        ["sauge", "salvia officinalis", "sage"]),
#    ("coriandre",               ["coriandre", "coriandrum sativum", "coriander"]),
#    ("aneth",                   ["aneth", "anethum graveolens", "dill"]),
#    #("fenouil",                 ["fenouil", "foeniculum vulgare", "fennel"]),
#    ("persil",                  ["persil", "petroselinum crispum", "parsley"]),
#    #("cerfeuil",                ["cerfeuil", "anthriscus cerefolium", "chervil"]),
#    ("ciboulette",              ["ciboulette", "allium schoenoprasum", "chives"]),
#    ("estragon",                ["estragon", "artemisia dracunculus", "tarragon"]),
#    #("laurier sauce",           ["laurier sauce", "laurus nobilis", "bay leaf"]),
#    #("melisse",                 ["melisse", "melissa officinalis",  "lemon balm"]),
#    #("liveche",                 ["liveche", "flevisticum officinale", "lovage"]),
#    #("cerfeuil musque",         ["cerfeuil musque", "sweet cicely", "myrrhis odorata"]),
#    #("fenugrec",                ["fenugrec", "trigonella foenum-graecum", "fenugreek"]),
#    #("carvi",                   ["carvi", "carum carvi", "caraway"]),
#    #("anis vert",               ["anis vert", "pimpinella anisum", "anise"]),
#    #("cumin",                   ["cumin", "cuminum cyminum", "cumin"]),
#    #("hysope officinale",       ["hysope officinale", "hyssopus officinalis", "hyssop"]),
#    #("agastache fenouil",       ["agastache fenouil", "agastache foeniculum", "anise hyssop"]),
#    #("verveine officinale",     ["verveine officinale", "verbena officinalis", "vervain"]),
#    #("bourrache",               ["bourrache", "borago officinalis", "borage"]),
#    #("souci officinal",         ["souci officinal", "calendula officinalis", "calendula"]),
#    #("camomille romaine",       ["camomille romaine", "chamaemelum nobile", "roman chamomile"]),
#]


# ─────────────────────────────────────────────
#  FONCTIONS
# ─────────────────────────────────────────────

def slug(name: str) -> str:
    replacements = {
        "é": "e", "è": "e", "ê": "e", "ë": "e",
        "à": "a", "â": "a", "ä": "a",
        "î": "i", "ï": "i",
        "ô": "o", "ö": "o",
        "ù": "u", "û": "u", "ü": "u",
        "ç": "c", " ": "_", "/": "-",
    }
    result = name.lower()
    for src, dst in replacements.items():
        result = result.replace(src, dst)
    return result


def fetch_image_urls(queries: list[str], count: int) -> list[str]:
    """
    Interroge DuckDuckGo Images via le package `ddgs`.
    API : ddgs.images(query, size=, type_image=, max_results=)
    """
    urls: list[str] = []
    seen: set[str]  = set()

    ddgs = DDGS()

    for query in queries:
        if len(urls) >= count:
            break

        needed = count - len(urls)
        try:
            results = ddgs.images(
                query,                              # ← positional, PAS keywords=
                size="Large",
                type_image="photo",
                max_results=min(needed + 30, 200),  # marge pour les échecs DL
            )
            for r in results:
                url = r.get("image", "")
                if url and url not in seen:
                    urls.append(url)
                    seen.add(url)
                if len(urls) >= count:
                    break

        except Exception as e:
            print(f"    [ERREUR DDG] requête '{query}' → {e}")

        time.sleep(1.0)  # pause entre requêtes DDG pour éviter le rate-limit

    return urls[:count]


def download_and_resize(url: str, dest_path: str, size: tuple[int, int]) -> bool:
    """Télécharge, crop centré et redimensionne en JPEG 400x400."""
    try:
        resp = requests.get(url, timeout=TIMEOUT_S, headers=HEADERS)
        resp.raise_for_status()

        img = Image.open(BytesIO(resp.content)).convert("RGB")

        w, h   = img.size
        target = size[0] / size[1]
        ratio  = w / h

        if ratio > target:
            new_w = int(h * target)
            left  = (w - new_w) // 2
            img   = img.crop((left, 0, left + new_w, h))
        elif ratio < target:
            new_h = int(w / target)
            top   = (h - new_h) // 2
            img   = img.crop((0, top, w, top + new_h))

        img = img.resize(size, Image.LANCZOS)
        img.save(dest_path, "JPEG", quality=90)
        return True

    except Exception as exc:
        print(f"    [ERREUR DL] {str(exc)[:80]}")
        return False


def process_aromate(
    common_name: str,
    queries: list[str],
    csv_writer,
    index: int,
    total: int,
) -> tuple[int, int]:
    folder_slug = slug(common_name)
    img_dir     = os.path.join(OUTPUT_DIR, folder_slug)
    os.makedirs(img_dir, exist_ok=True)

    print(f"\n{'═'*60}")
    print(f"[{index}/{total}] {common_name}")
    print(f"  Requêtes : {queries[0]} (+{len(queries)-1} fallbacks)")
    print(f"{'─'*60}")

    print("  Récupération des URLs…")
    urls = fetch_image_urls(queries, TARGET_COUNT)
    print(f"  → {len(urls)} URLs trouvées.")

    if not urls:
        print("  Aucune URL — aromate ignoré.")
        return 0, 0

    success = 0
    for i, url in enumerate(urls, start=1):
        filename  = f"{folder_slug}_{i:04d}.jpg"
        dest_path = os.path.join(img_dir, filename)
        rel_path  = os.path.join(OUTPUT_DIR, folder_slug, filename)

        print(f"    [{i:>3}/{len(urls)}] {filename} … ", end="", flush=True)

        if download_and_resize(url, dest_path, IMAGE_SIZE):
            csv_writer.writerow({"nom_aromate": common_name, "chemin_image": rel_path})
            success += 1
            print("OK")
        else:
            print("ECHEC")

        time.sleep(DELAY_S)

    return success, len(urls)


# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, CSV_FILENAME)

    total_ok    = 0
    total_tried = 0
    start_time  = time.time()

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["nom_aromate", "chemin_image"])
        writer.writeheader()

        for idx, (common, queries) in enumerate(AROMATES, start=1):
            ok, tried = process_aromate(common, queries, writer, idx, len(AROMATES))
            total_ok    += ok
            total_tried += tried
            f.flush()

    elapsed = int(time.time() - start_time)
    h, m, s = elapsed // 3600, (elapsed % 3600) // 60, elapsed % 60

    print(f"\n{'═'*60}")
    print(f"TERMINÉ — {total_ok}/{total_tried} images téléchargées")
    print(f"Durée totale : {h:02d}h{m:02d}m{s:02d}s")
    print(f"CSV          : {csv_path}")
    print(f"Dossier      : {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()


════════════════════════════════════════════════════════════
[1/10] thym commun
  Requêtes : thyme (+0 fallbacks)
────────────────────────────────────────────────────────────
  Récupération des URLs…
  → 92 URLs trouvées.
    [  1/92] thym_commun_0001.jpg … OK
    [  2/92] thym_commun_0002.jpg … OK
    [  3/92] thym_commun_0003.jpg … OK
    [  4/92] thym_commun_0004.jpg … OK
    [  5/92] thym_commun_0005.jpg … OK
    [  6/92] thym_commun_0006.jpg …     [ERREUR DL] 403 Client Error: Forbidden for url: https://paradisegarden.com.cy/wp-content/up
ECHEC
    [  7/92] thym_commun_0007.jpg … OK
    [  8/92] thym_commun_0008.jpg … OK
    [  9/92] thym_commun_0009.jpg … OK
    [ 10/92] thym_commun_0010.jpg … OK
    [ 11/92] thym_commun_0011.jpg … OK
    [ 12/92] thym_commun_0012.jpg … OK
    [ 13/92] thym_commun_0013.jpg … OK
    [ 14/92] thym_commun_0014.jpg … OK
    [ 15/92] thym_commun_0015.jpg … OK
    [ 16/92] thym_commun_0016.jpg … OK
    [ 17/92] thym_commun_0017.jpg … OK
    [ 18/92] t

/home/jouell/.pyenv/versions/3.11.8/envs/plant/lib/python3.11/site-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


OK
    [ 68/92] thym_commun_0068.jpg …     [ERREUR DL] HTTPSConnectionPool(host='hosnaexport.com', port=443): Max retries exceeded with
ECHEC
    [ 69/92] thym_commun_0069.jpg … OK
    [ 70/92] thym_commun_0070.jpg … OK
    [ 71/92] thym_commun_0071.jpg … OK
    [ 72/92] thym_commun_0072.jpg … OK
    [ 73/92] thym_commun_0073.jpg … OK
    [ 74/92] thym_commun_0074.jpg … OK
    [ 75/92] thym_commun_0075.jpg … OK
    [ 76/92] thym_commun_0076.jpg … OK
    [ 77/92] thym_commun_0077.jpg … OK
    [ 78/92] thym_commun_0078.jpg … OK
    [ 79/92] thym_commun_0079.jpg … OK
    [ 80/92] thym_commun_0080.jpg … OK
    [ 81/92] thym_commun_0081.jpg … OK
    [ 82/92] thym_commun_0082.jpg … OK
    [ 83/92] thym_commun_0083.jpg … OK
    [ 84/92] thym_commun_0084.jpg … OK
    [ 85/92] thym_commun_0085.jpg … OK
    [ 86/92] thym_commun_0086.jpg … OK
    [ 87/92] thym_commun_0087.jpg … OK
    [ 88/92] thym_commun_0088.jpg … OK
    [ 89/92] thym_commun_0089.jpg … OK
    [ 90/92] thym_commun_0090.jpg … OK
